# 🏦 try-self — EDA & Training Walkthrough

This notebook mirrors the `src/` pipeline interactively:
1. Load & clean the loan dataset
2. Explore key drivers of approval
3. Train and compare models
4. Inspect an individual prediction


In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from src.preprocess import prepare, feature_columns, TARGET

df = prepare(pd.read_csv('../data/raw/loan_data.csv'))
df.head()


## 1. Quick EDA


In [ ]:
print(f'Rows: {len(df)} | Approval rate: {df[TARGET].mean():.2%}')
df.describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby('credit_history')[TARGET].mean().plot.bar(ax=axes[0], title='Approval rate by credit history')
df.groupby('property_area')[TARGET].mean().plot.bar(ax=axes[1], title='Approval rate by property area')
plt.tight_layout()


In [ ]:
df.plot.scatter(x='dti_ratio', y=TARGET, alpha=0.15, figsize=(7, 3), title='Debt-to-income vs approval');


## 2. Train & compare models

The same logic as `python -m src.train` — best model by ROC-AUC is persisted.


In [ ]:
from src.train import build_candidates, evaluate
from sklearn.model_selection import train_test_split

X, y = df[feature_columns()], df[TARGET].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

for name, pipe in build_candidates().items():
    pipe.fit(X_tr, y_tr)
    print(f'{name:>22}: {evaluate(pipe, X_te, y_te)}')


## 3. Explain one prediction


In [ ]:
from src.predict import predict_application

predict_application({
    'applicant_income': 5400, 'coapplicant_income': 1200,
    'loan_amount': 128000, 'loan_term_months': 360,
    'credit_history': 1, 'dependents': 0,
    'employment_status': 'salaried', 'property_area': 'urban',
})


## Next steps
- Run `python -m src.evaluate` for threshold and fairness reports
- Launch the API: `uvicorn src.api:app --reload`
- Launch the demo UI: `streamlit run app.py`
